In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import torchaudio
import torchaudio.transforms as T
import numpy as np
from tqdm import tqdm

# --- STRICT GPU CHECK ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Initializing Phase 4 on: {device}")
if device.type == 'cpu':
    print("⚠️ WARNING: You are on CPU. Stop the session and switch Accelerator to GPU T4 x2!")

# --- ARCHITECTURE DEFINITIONS ---
class VideoBranch(nn.Module):
    def __init__(self):
        super(VideoBranch, self).__init__()
        self.backbone = timm.create_model('xception', pretrained=False, num_classes=0)
        self.lstm = nn.LSTM(input_size=self.backbone.num_features, hidden_size=512, batch_first=True)

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.size()
        x = x.view(batch_size * seq_length, c, h, w)
        features = self.backbone.forward_features(x)
        features = nn.functional.adaptive_avg_pool2d(features, (1, 1)).view(features.size(0), -1) 
        lstm_out, _ = self.lstm(features.view(batch_size, seq_length, -1))
        return lstm_out[:, -1, :] # Returns 512-dim vector

class AudioBranch(nn.Module):
    def __init__(self):
        super(AudioBranch, self).__init__()
        self.backbone = timm.create_model('resnet18', pretrained=False, in_chans=1, num_classes=0)

    def forward(self, x):
        return self.backbone(x) # Returns 512-dim vector

class MultimodalFusionModel(nn.Module):
    def __init__(self, video_weights_path, audio_weights_path, device):
        super(MultimodalFusionModel, self).__init__()
        self.video_net = VideoBranch().to(device)
        self.audio_net = AudioBranch().to(device)
        
        print("Loading pre-trained expert weights...")
        self.video_net.load_state_dict(torch.load(video_weights_path, map_location=device), strict=False)
        self.audio_net.load_state_dict(torch.load(audio_weights_path, map_location=device), strict=False)
        
        # Freeze the branches! We only want to train the fusion layer.
        for param in self.video_net.parameters(): param.requires_grad = False
        for param in self.audio_net.parameters(): param.requires_grad = False
        
        # Late Fusion Classifier (512 + 512 = 1024 -> 1)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, video_seq, audio_spec):
        v_feat = self.video_net(video_seq)
        a_feat = self.audio_net(audio_spec)
        combined = torch.cat((v_feat, a_feat), dim=1)
        return self.classifier(combined)

In [ ]:
import os

print("🔍 Inspecting Frames Directory...")
frames_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'frames' in dirs:
        frames_dir = os.path.join(root, 'frames')
        break

if not frames_dir:
    print("❌ ERROR: Could not find a 'frames' folder anywhere in /kaggle/input!")
else:
    print(f"📁 Found frames folder at: {frames_dir}")
    
    count = 0
    for r, d, f in os.walk(frames_dir):
        jpgs = [img for img in f if img.endswith(('.jpg', '.png'))]
        if len(jpgs) > 0:
            print("-" * 30)
            print(f"Sample Path: {r}")
            print(f"Extracted Folder Name: '{os.path.basename(r)}'")
            print(f"Extracted Parent Name: '{os.path.basename(os.path.dirname(r))}'")
            count += 1
            if count >= 3: 
                break
                
    if count == 0:
        print("❌ ERROR: Found the 'frames' folder, but there are ZERO .jpg files inside it!")

In [ ]:
import os
import torch
import cv2
import torchaudio
import torchaudio.transforms as T
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

# --- 1. LOCATE THE VIDEOS ONLY ---
videos_input = '/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2'

print(f"🚀 Initializing Direct-from-MP4 Dataloader...")

class UltimateFusionDataset(Dataset):
    def __init__(self, videos_root, sequence_length=10):
        self.sequence_length = sequence_length
        self.samples = []
        
        # Notice we added ToPILImage so OpenCV arrays play nice with your transforms
        self.v_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((299, 299)), 
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

        print(f"Scanning for MP4s in {videos_root}...")
        for root, dirs, files in os.walk(videos_root):
            for file in files:
                if file.endswith('.mp4'):
                    path = os.path.join(root, file)
                    # Ground truth logic directly from the folder name
                    is_fake = 'FakeVideo' in root or 'FakeAudio' in root
                    self.samples.append((path, 1.0 if is_fake else 0.0))
                    
        print(f"✅ Locked in {len(self.samples)} videos. No folder matching needed!")

    def __len__(self): 
        return len(self.samples)

    def __getitem__(self, idx):
        mp4_path, label = self.samples[idx]
        
        # --- 1. VIDEO EXTRACTION ON THE FLY ---
        cap = cv2.VideoCapture(mp4_path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames > 0:
            step = max(1, total_frames // self.sequence_length)
            for i in range(self.sequence_length):
                cap.set(cv2.CAP_PROP_POS_FRAMES, min(i * step, total_frames - 1))
                ret, frame = cap.read()
                if ret:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(self.v_transform(frame))
                else:
                    frames.append(torch.zeros((3, 299, 299))) # Fallback
        else:
            # Fallback for completely corrupted video files
            frames = [torch.zeros((3, 299, 299)) for _ in range(self.sequence_length)]
            
        cap.release()
        v_tensor = torch.stack(frames)
        
        # --- 2. AUDIO EXTRACTION ON THE FLY ---
        try:
            waveform, sr = torchaudio.load(mp4_path)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            if sr != 16000: waveform = T.Resample(sr, 16000)(waveform)
            num_samples = 16000 * 3
            waveform = waveform[:, :num_samples] if waveform.shape[1] > num_samples else torch.nn.functional.pad(waveform, (0, num_samples - waveform.shape[1]))
            spec = T.AmplitudeToDB()(T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=64)(waveform))
        except:
            spec = torch.zeros((1, 64, 94)) # Fallback
        
        return v_tensor, spec, torch.tensor([label], dtype=torch.float32)

# --- 2. INITIALIZATION ---
full_multimodal_dataset = UltimateFusionDataset(videos_input)

if len(full_multimodal_dataset) > 0:
    indices = list(range(len(full_multimodal_dataset)))
    np.random.seed(42)
    np.random.shuffle(indices)
    split = int(0.2 * len(indices))
    train_idx, val_idx = indices[split:], indices[:split]

    # Using num_workers=2 to help the CPU extract video frames faster
    train_loader = DataLoader(Subset(full_multimodal_dataset, train_idx), batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(Subset(full_multimodal_dataset, val_idx), batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
    print(f"✅ Ready for training: {len(train_idx)} Train | {len(val_idx)} Val")
else:
    print("❌ ERROR: Could not find any .mp4 files. Check the videos_input path.")

In [ ]:
# --- MISSING INITIALIZATION STEP ---
print("⚙️ Initializing Multimodal Fusion Model...")

# Your exact paths from the earlier successful scan
v_pth = '/kaggle/input/datasets/abhishekk011/previous-phase/best_temporal_lstm_model (1).pth'
a_pth = '/kaggle/input/datasets/abhishekk011/previous-phase/best_audio_model.pth'

# Initialize and move to GPU
fusion_model = MultimodalFusionModel(v_pth, a_pth, device).to(device)

print("✅ fusion_model successfully defined and loaded to GPU!")

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(fusion_model.classifier.parameters(), lr=1e-4)

print("🚀 Starting Final Multimodal Fusion Training...")
num_epochs = 5

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print('-' * 10)
    
    # --- TRAINING ---
    fusion_model.train()
    running_loss, correct_train = 0.0, 0
    
    for v_seq, a_spec, labels in tqdm(train_loader, desc="Training Fusion"):
        v_seq, a_spec, labels = v_seq.to(device), a_spec.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = fusion_model(v_seq, a_spec)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * v_seq.size(0)
        preds = (outputs > 0.5).float()
        correct_train += (preds == labels).sum().item()
        
    train_acc = correct_train / len(train_idx)
    
    # --- VALIDATION ---
    fusion_model.eval()
    val_loss, correct_val = 0.0, 0
    with torch.no_grad():
        for v_seq, a_spec, labels in tqdm(val_loader, desc="Validating"):
            v_seq, a_spec, labels = v_seq.to(device), a_spec.to(device), labels.to(device)
            outputs = fusion_model(v_seq, a_spec)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * v_seq.size(0)
            preds = (outputs > 0.5).float()
            correct_val += (preds == labels).sum().item()
    
    val_acc = correct_val / len(val_idx)
    print(f"Train Loss: {running_loss/len(train_idx):.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss/len(val_idx):.4f} | Val Acc:   {val_acc:.4f}")

torch.save(fusion_model.state_dict(), 'final_multimodal_deepfake_detector.pth')
print("✅ ALL PHASES COMPLETE. Final thesis model saved successfully.")

In [ ]:
import torch
torch.save(fusion_model.state_dict(), '/kaggle/working/final_multimodal_deepfake_detector.pth')
print("✅ Manual save complete! Please refresh the Kaggle output folder.")